In [4]:
import regex
import json
from typing import Iterable, Iterator
import pickle
from pathlib import Path

class Tokenizer:
    def __init__(
        self, vocab: dict[int, bytes], 
        merges: list[tuple[bytes, bytes]], 
        special_tokens: list[str] | None = None
        ):
        '''从给定的词表、合并规则和（可选的）特殊 tokens 构造一个分词器'''
        self.vocab = vocab
        self.merges = merges
        self.vocab_inverse = {v: k for k, v in vocab.items()}
        self.merges_ranked = {(k[0], k[1]): v for v, k in enumerate(merges)}
        self.special_tokens = special_tokens
        
        # 将词典中没有的特殊token添加到词典中
        if self.special_tokens:
            for n, t in enumerate(self.special_tokens):
                if t.encode() not in self.vocab_inverse:
                    new_idx = len(self.vocab) + 1 + n
                    self.vocab_inverse[t] = new_idx
                    self.vocab[new_idx] = t.encode()

    @classmethod
    def from_files(cls, vocab_filepath: str, 
                   merges_filepath: str, 
                   special_tokens: list[str] | None = None):
        '''从序列化的 vocab 和 merges 文件构造一个 Tokenizer'''
        # 读取vocab
        if vocab_filepath[-4:] == '.pkl':
            vocab_path = Path(vocab_filepath)
            with vocab_path.open("rb") as f:
                vocab = pickle.load(f)
                
        elif vocab_filepath[-4:] == 'json':
            with open(vocab_filepath, 'rb') as f:
                vocab = json.load(f)

        # 读取merges
        merges = []
        with open(merges_filepath) as f:
            for line in f:
                cleaned_line = line.rstrip()
                if cleaned_line and len(cleaned_line.split(" ")) == 2:
                    merges.append(tuple(cleaned_line.split(" ")))
            return cls(vocab, merges, special_tokens)
    
    def encode(self, text: str) -> list[int]:
        '''将输入文本编码为 token ID 序列'''
        # 0. 按照特殊token对文本进行分段
        if self.special_tokens:
            # 这里主要是解决同时出现<|endoftext|><|endoftext|>的情况
            sorted_special_tokens = sorted(self.special_tokens, key=len, reverse=True)
            special_token_pattern = '|'.join(map(regex.escape, sorted_special_tokens))
            chunks = regex.split(f'({special_token_pattern})', text)
        else:
            chunks = [text]
        idx = []
        
        # 对于每一段文本
        for chunk in chunks:
            if not chunk: continue
            if self.special_tokens is not None and chunk in self.special_tokens:
                idx.append(self.vocab_inverse[chunk.encode()])
            else:
                # 1. 预分词
                PAT = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
                pre_tokens = regex.findall(PAT, chunk)
                # 2. 对每个token应用BPE合并
                for pre_token in pre_tokens:
                    # 将 pre_token 的内容进行 utf-8 编码
                    pre_token = [bytes([i]) for i in pre_token.encode()]
                    while True:
                        new_pre_token = []
                        to_merge = dict()
                        for index1, index2 in zip(pre_token, pre_token[1:]):
                            if (index1, index2) in self.merges_ranked:
                                to_merge[(index1, index2)] = self.merges_ranked[(index1, index2)]
                        if len(to_merge) == 0:
                            break
                        # 找到合并优先级最高的
                        pair = min(to_merge, key=to_merge.get)
                        # 合并
                        i = 0
                        while i < len(pre_token):
                            if i + 1 < len(pre_token) and (pre_token[i], pre_token[i + 1]) == pair:
                                new_pre_token.append(pair[0] + pair[1])
                                i += 2
                            else:
                                new_pre_token.append(pre_token[i])
                                i += 1
                        pre_token = new_pre_token.copy()
                    for i in pre_token:
                        if i in self.vocab_inverse:
                            idx.append(self.vocab_inverse[i])
                        else:
                            idx.append(ord(i))
        return idx

    def encode_iterable(self, iterable: Iterable[str]) -> Iterator[int]:
        '''给定字符串的可迭代对象（如文件句柄），惰性地产生 token IDs'''
        for text in iterable:
            for tid in self.encode(text):
                yield tid
        
    def decode(self, ids: list[int]) -> str:
        '''将 token ID 序列解码回原始文本'''
        byte_sequence = b''
        for i in ids:
            if i in self.vocab:
                byte_sequence += self.vocab[i]
            else:
                byte_sequence += b'\xff'
        # 使用 errors="replace" 保证非法字节能被替换成 �
        return byte_sequence.decode("utf-8", errors="replace")
    
def get_tokenizer(
    vocab: dict[int, bytes],
    merges: list[tuple[bytes, bytes]],
    special_tokens: list[str] | None = None,):
    tokenizer = Tokenizer(vocab, merges, special_tokens)
    return tokenizer

In [71]:
import os
from tests.common import FIXTURES_PATH, gpt2_bytes_to_unicode

VOCAB_PATH = FIXTURES_PATH / "gpt2_vocab.json"
MERGES_PATH = FIXTURES_PATH / "gpt2_merges.txt"

def gpt2_bytes_to_unicode() -> dict[int, str]:

    # These 188 integers can used as-is, since they are not whitespace or control characters.
    # See https://www.ssec.wisc.edu/~tomw/java/unicode.html.
    bs = list(range(ord("!"), ord("~") + 1)) + list(range(ord("¡"), ord("¬") + 1)) + list(range(ord("®"), ord("ÿ") + 1))
    cs = bs[:]
    # now get the representations of the other 68 integers that do need shifting
    # each will get mapped chr(256 + n), where n will grow from 0...67 in the loop
    # Get printable representations of the remaining integers 68 integers.
    n = 0
    for b in range(2**8):
        if b not in bs:
            # If this integer isn't in our list of visually-representable
            # charcters, then map it to the next nice character (offset by 256)
            bs.append(b)
            cs.append(2**8 + n)
            n += 1
    characters = [chr(n) for n in cs]
    d = dict(zip(bs, characters))
    return d


def get_tokenizer_from_vocab_merges_path(
    vocab_path: str | os.PathLike,
    merges_path: str | os.PathLike,
    special_tokens: list[str] | None = None,
):
    gpt2_byte_decoder = {v: k for k, v in gpt2_bytes_to_unicode().items()}
    with open(vocab_path) as vocab_f:
        gpt2_vocab = json.load(vocab_f)
    gpt2_bpe_merges = []
    with open(merges_path) as f:
        for line in f:
            cleaned_line = line.rstrip()
            if cleaned_line and len(cleaned_line.split(" ")) == 2:
                gpt2_bpe_merges.append(tuple(cleaned_line.split(" ")))
    # The GPT-2 tokenizer uses a remapped unicode encoding for bytes. Let's
    # just return the original bytes, so we don't force students to use
    # any particular encoding scheme.
    vocab = {
        gpt2_vocab_index: bytes([gpt2_byte_decoder[token] for token in gpt2_vocab_item])
        for gpt2_vocab_item, gpt2_vocab_index in gpt2_vocab.items()
    }
    # If any of the special tokens don't exist in the vocab, append them to the vocab.
    if special_tokens:
        for special_token in special_tokens:
            byte_encoded_special_token = special_token.encode("utf-8")
            if byte_encoded_special_token not in set(vocab.values()):
                vocab[len(vocab)] = byte_encoded_special_token

    merges = [
        (
            bytes([gpt2_byte_decoder[token] for token in merge_token_1]),
            bytes([gpt2_byte_decoder[token] for token in merge_token_2]),
        )
        for merge_token_1, merge_token_2 in gpt2_bpe_merges
    ]
    return get_tokenizer(vocab, merges, special_tokens)

In [115]:
# 测试单个字符
def test_roundtrip_single_character():
    tokenizer = get_tokenizer_from_vocab_merges_path(
        vocab_path=VOCAB_PATH,
        merges_path=MERGES_PATH,
    )
    test_string = "s"
    encoded_ids = tokenizer.encode(test_string)
    decoded_string = tokenizer.decode(encoded_ids)
    print(encoded_ids)
    print(decoded_string)
    assert test_string == decoded_string

test_roundtrip_single_character()

[82]
s


In [114]:
def test_roundtrip_single_character():
    tokenizer = get_tokenizer_from_vocab_merges_path(
        vocab_path=VOCAB_PATH,
        merges_path=MERGES_PATH,
    )
    test_string = "s"
    encoded_ids = tokenizer.encode(test_string)
    decoded_string = tokenizer.decode(encoded_ids)
    assert test_string == decoded_string
    
test_roundtrip_single_character()

In [113]:

import tiktoken
def test_single_character_matches_tiktoken():
    reference_tokenizer = tiktoken.get_encoding("gpt2")
    tokenizer = get_tokenizer_from_vocab_merges_path(
        vocab_path=VOCAB_PATH,
        merges_path=MERGES_PATH,
    )
    test_string = "s"

    reference_ids = reference_tokenizer.encode(test_string)
    ids = tokenizer.encode(test_string)
    print('reference_ids: ', reference_ids)
    print('ids: ', ids)
    assert ids == reference_ids

    tokenized_string = [tokenizer.decode([x]) for x in ids]
    assert tokenized_string == ["s"]

    assert tokenizer.decode(ids) == test_string
    assert reference_tokenizer.decode(reference_ids) == test_string
test_single_character_matches_tiktoken()

reference_ids:  [82]
ids:  [82]


In [112]:
def test_single_unicode_character_matches_tiktoken():
    reference_tokenizer = tiktoken.get_encoding("gpt2")
    tokenizer = get_tokenizer_from_vocab_merges_path(
        vocab_path=VOCAB_PATH,  
        merges_path=MERGES_PATH,
    )
    test_string = "🙃"

    reference_ids = reference_tokenizer.encode(test_string)
    ids = tokenizer.encode(test_string)
    
    print(reference_ids)
    print(ids)

    assert ids == reference_ids

    assert tokenizer.decode(ids) == test_string
    assert reference_tokenizer.decode(reference_ids) == test_string

test_single_unicode_character_matches_tiktoken()

[8582, 247, 225]
[8582, 247, 225]


In [134]:
def test_roundtrip_unicode_string_with_special_tokens():
    tokenizer = get_tokenizer_from_vocab_merges_path(
        vocab_path=VOCAB_PATH, merges_path=MERGES_PATH, special_tokens=["<|endoftext|>"]
    )
    test_string = "Héllò hôw <|endoftext|><|endoftext|> are ü? 🙃<|endoftext|>"
    encoded_ids = tokenizer.encode(test_string)
    print(encoded_ids)
    tokenized_string = [tokenizer.decode([x]) for x in encoded_ids]

    print(tokenized_string)
    print(encoded_ids)

    # Ensure the special <|endoftext|> token is preserved
    assert tokenized_string.count("<|endoftext|>") == 3

    decoded_string = tokenizer.decode(encoded_ids)
    assert test_string == decoded_string
    
test_roundtrip_unicode_string_with_special_tokens()

[39, 2634, 297, 127, 110, 289, 27083, 86, 220, 50258, 50258, 389, 6184, 120, 30, 12520, 247, 225, 50258]
self.vocab[i]:  b'H'
byte_sequence: b'H'
self.vocab[i]:  b'\xc3\xa9'
byte_sequence: b'\xc3\xa9'
self.vocab[i]:  b'll'
byte_sequence: b'll'
self.vocab[i]:  b'\xc3'
byte_sequence: b'\xc3'
self.vocab[i]:  b'\xb2'
byte_sequence: b'\xb2'
self.vocab[i]:  b' h'
byte_sequence: b' h'
self.vocab[i]:  b'\xc3\xb4'
byte_sequence: b'\xc3\xb4'
self.vocab[i]:  b'w'
byte_sequence: b'w'
self.vocab[i]:  b' '
byte_sequence: b' '
self.vocab[i]:  b'<|endoftext|>'
byte_sequence: b'<|endoftext|>'
self.vocab[i]:  b'<|endoftext|>'
byte_sequence: b'<|endoftext|>'
self.vocab[i]:  b' are'
byte_sequence: b' are'
self.vocab[i]:  b' \xc3'
byte_sequence: b' \xc3'
self.vocab[i]:  b'\xbc'
byte_sequence: b'\xbc'
self.vocab[i]:  b'?'
byte_sequence: b'?'
self.vocab[i]:  b' \xf0\x9f'
byte_sequence: b' \xf0\x9f'
self.vocab[i]:  b'\x99'
byte_sequence: b'\x99'
self.vocab[i]:  b'\x83'
byte_sequence: b'\x83'
self.vocab[i]:  b

In [147]:
def test_unicode_string_with_special_tokens_matches_tiktoken():
    reference_tokenizer = tiktoken.get_encoding("gpt2")
    tokenizer = get_tokenizer_from_vocab_merges_path(
        vocab_path=VOCAB_PATH, merges_path=MERGES_PATH, special_tokens=["<|endoftext|>"]
    )
    test_string = "Héllò hôw <|endoftext|><|endoftext|> are ü? 🙃<|endoftext|>"

    reference_ids = reference_tokenizer.encode(test_string, allowed_special={"<|endoftext|>"})
    ids = tokenizer.encode(test_string)
    
    print(reference_ids)
    print(ids)
    
    assert ids == reference_ids

    assert tokenizer.decode(ids) == test_string
    assert reference_tokenizer.decode(reference_ids) == test_string
    
test_unicode_string_with_special_tokens_matches_tiktoken()

[39, 2634, 297, 127, 110, 289, 27083, 86, 220, 50256, 50256, 389, 6184, 120, 30, 12520, 247, 225, 50256]
[39, 2634, 297, 127, 110, 289, 27083, 86, 220, 50256, 50256, 389, 6184, 120, 30, 12520, 247, 225, 50256]
self.vocab[i]:  b'H'
byte_sequence: b'H'
self.vocab[i]:  b'\xc3\xa9'
byte_sequence: b'H\xc3\xa9'
self.vocab[i]:  b'll'
byte_sequence: b'H\xc3\xa9ll'
self.vocab[i]:  b'\xc3'
byte_sequence: b'H\xc3\xa9ll\xc3'
self.vocab[i]:  b'\xb2'
byte_sequence: b'H\xc3\xa9ll\xc3\xb2'
self.vocab[i]:  b' h'
byte_sequence: b'H\xc3\xa9ll\xc3\xb2 h'
self.vocab[i]:  b'\xc3\xb4'
byte_sequence: b'H\xc3\xa9ll\xc3\xb2 h\xc3\xb4'
self.vocab[i]:  b'w'
byte_sequence: b'H\xc3\xa9ll\xc3\xb2 h\xc3\xb4w'
self.vocab[i]:  b' '
byte_sequence: b'H\xc3\xa9ll\xc3\xb2 h\xc3\xb4w '
self.vocab[i]:  b'<|endoftext|>'
byte_sequence: b'H\xc3\xa9ll\xc3\xb2 h\xc3\xb4w <|endoftext|>'
self.vocab[i]:  b'<|endoftext|>'
byte_sequence: b'H\xc3\xa9ll\xc3\xb2 h\xc3\xb4w <|endoftext|><|endoftext|>'
self.vocab[i]:  b' are'
byte_sequence

In [5]:
import pickle
from pathlib import Path
import ast

# 读取10行文本
data_path = Path("../data/TinyStoriesV2-GPT4-valid.txt")
delimiter = "<|endoftext|>"
stories = []
with data_path.open("r", encoding="utf-8") as f:
    buffer = []
    for line in f:
        if line.strip() == delimiter:
            story = "".join(buffer).strip()
            if story:
                stories.append(story)
            buffer = []
            if len(stories) == 10:
                break
        else:
            buffer.append(line)
stories

['Spot. Spot saw the shiny car and said, "Wow, Kitty, your car is so bright and clean!" Kitty smiled and replied, "Thank you, Spot. I polish it every day."\nAfter playing with the car, Kitty and Spot felt thirsty. They found a small pond with clear water. They drank the water and felt very happy. They played together all day and became best friends.',
 'Once upon a time, in a big forest, there lived a rhinoceros named Roxy. Roxy loved to climb. She climbed trees, rocks, and hills. One day, Roxy found an icy hill. She had never seen anything like it before. It was shiny and cold, and she wanted to climb it.\nRoxy tried to climb the icy hill, but it was very slippery. She tried again and again, but she kept falling down. Roxy was sad. She wanted to climb the icy hill so much. Then, she saw a little bird named Billy. Billy saw that Roxy was sad and asked, "Why are you sad, Roxy?"\nRoxy told Billy about the icy hill and how she couldn\'t climb it. Billy said, "I have an idea! Let\'s find s

In [6]:
# 读取 vocab 和 merges
vocab_path = Path("result/TinyStories_vocab.pkl")
with vocab_path.open("rb") as f:
    vocab = pickle.load(f)

merges_path = Path("result/TinyStories_merges.txt")
merges = []
with merges_path.open("r", encoding="utf-8") as f:
    for line in f:
        raw = line.strip()
        if not raw or raw.startswith("#"):
            continue
        pair = ast.literal_eval(raw)  # yields (b'h', b'e')
        if not isinstance(pair, tuple) or len(pair) != 2:
            raise ValueError(f"Unexpected merge line: {raw}")
        merges.append(pair)

# Tokenizer 实例化
tokenizer_TS = Tokenizer(vocab=vocab, merges=merges)

# 计算压缩率
result_bytes = b''
result_token = []
for i in stories:
    token_story = tokenizer_TS.encode(i)
    bytes_story = i.encode()
    result_bytes += bytes_story
    result_token += token_story
print(len(result_bytes) / len(result_token))

4.034077555816686


In [7]:
import time
with open("../data/TinyStoriesV2-GPT4-valid.txt", "rb") as f:
    first_gig = f.read()
text_part = first_gig.decode("utf-8", errors="ignore")
tokenizer_TS = Tokenizer(vocab, merges, ["<|endoftext|>"])

# 开始计时
time_start = time.time()
tokenizer_TS.encode(text_part)
time_end = time.time()

times = time_end - time_start
times

42.40232038497925

In [8]:
times * 825

34981.91431760788

In [9]:
times * 825 / 60 / 60

9.717198421557745

In [ ]:
import numpy as np
import sys 

# 测试集编码
with open('../data/TinyStoriesV2-GPT4-valid.txt', 'r') as f:
    TS_valid = f.read()
TS_valid_token = tokenizer_TS.encode(TS_valid)
# 保存
arr = np.array(TS_valid_token, dtype=np.uint16)
np.save("result/TS_valid_token.npy", arr)

# 训练集编码
with open('../data/TinyStoriesV2-GPT4-train.txt', 'r') as f:
    TS_train = f.read()
TS_train_token = tokenizer_TS.encode(TS_train)
# 保存
arr = np.array(TS_train_token, dtype=np.uint16)
np.save("result/TS_train_token.npy", arr)

In [ ]:
print('列表大小：',sys.getsizeof(TS_train_token) / 1024 / 1024)
loaded = np.load("result/TS_train_token.npy")
print('用uint16存储大小：',sys.getsizeof(loaded) / 1024 / 1024)

In [23]:
loaded = np.load("result/TS_train_token.npy")
sys.getsizeof(loaded) / 1024 / 1024

880.9276714324951